In [ ]:
# ============================================
# MODULE 1: Decision Extractor (CLEAN RESET)
# ============================================

import requests
import json
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_NEW_API_KEY")
MODEL = "llama-3.3-70b-versatile"

def extract_decision(text: str) -> dict:
    prompt = f"""You are a corporate decision intelligence engine.
Extract structured decision data from the following text.
Return ONLY valid JSON, no extra text:
{{
  "decision": "core decision made",
  "rationale": "why this decision was made",
  "assumptions": ["assumption1", "assumption2"],
  "owner": "person/team responsible",
  "deadline": "deadline if mentioned, else null",
  "domain": "HR/Finance/Tech/Operations/Strategy/Other",
  "keywords": ["kw1", "kw2", "kw3"]
}}

Text: {text}"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        },
        timeout=60
    )

    resp_json = response.json()
    if "choices" not in resp_json:
        print("❌ API ERROR:", resp_json)
        raise ValueError(f"Bad response: {resp_json}")

    raw = resp_json["choices"][0]["message"]["content"]
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

# TEST
sample = """
The leadership team has decided to freeze all external hiring for Q3 2025.
This is driven by budget overruns in Q2. The CFO owns this decision.
All department heads must comply by July 1st.
"""
result = extract_decision(sample)
print(json.dumps(result, indent=2))

In [ ]:
# ============================================
# MODULE 2: Embedding + ChromaDB Storage
# ============================================

!pip install chromadb sentence-transformers -q

import chromadb
import uuid
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("decisions")

def flatten_metadata(obj: dict) -> dict:
    """Convert all list fields to strings for ChromaDB compatibility."""
    return {k: (", ".join(v) if isinstance(v, list) else (v or "N/A")) for k, v in obj.items()}

def store_decision(decision_obj: dict) -> str:
    decision_id = str(uuid.uuid4())
    text_to_embed = f"{decision_obj['decision']}. {decision_obj['rationale']}. {' '.join(decision_obj['keywords'])}"
    embedding = embedder.encode(text_to_embed).tolist()

    collection.add(
        ids=[decision_id],
        embeddings=[embedding],
        documents=[text_to_embed],
        metadatas=[flatten_metadata(decision_obj)]
    )
    print(f"✅ Stored Decision ID: {decision_id}")
    return decision_id

def retrieve_similar(query: str, top_k: int = 5) -> list:
    embedding = embedder.encode(query).tolist()
    results = collection.query(query_embeddings=[embedding], n_results=top_k)
    return results["metadatas"][0]


# ── TEST ──────────────────────────────────
did = store_decision(result)
similar = retrieve_similar("hiring budget decision")
print("\n🔍 Similar Decisions:")
for d in similar:
    print(f" → {d['decision']} | Domain: {d['domain']}")

# ── SEED MORE DECISIONS FOR TESTING ──────
samples = [
    """Marketing team decided to increase hiring budget by 40% in Q3 2025
    to accelerate product launches. CMO owns this. Deadline: June 15th.""",

    """Engineering decided to migrate all infrastructure to AWS by Q4 2025
    to reduce costs. CTO owns this. Deadline: December 1st.""",

    """Finance has decided to cut all department budgets by 25% for Q3 2025
    due to revenue shortfall. CFO owns this. Effective immediately."""
]

stored_ids = [did]
for s in samples:
    extracted = extract_decision(s)
    print(json.dumps(extracted, indent=2))
    sid = store_decision(extracted)
    stored_ids.append(sid)

print(f"\n✅ Total decisions stored: {collection.count()}")

In [ ]:
# ============================================
# MODULE 3: Contradiction & Dependency Detector
# ============================================

GROQ_API_KEY = userdata.get("GROQ_NEW_API_KEY").strip()

def detect_contradictions(new_decision_obj: dict) -> list:
    query = f"{new_decision_obj['decision']}. {new_decision_obj['rationale']}"
    similar = retrieve_similar(query, top_k=5)

    if not similar:
        return []

    similar_text = "\n".join([
        f"- [{i+1}] Decision: {d['decision']} | Rationale: {d['rationale']} | Domain: {d['domain']} | Owner: {d['owner']}"
        for i, d in enumerate(similar)
    ])

    prompt = f"""You are a corporate decision auditor.

A new decision has been made:
Decision: {new_decision_obj['decision']}
Rationale: {new_decision_obj['rationale']}
Domain: {new_decision_obj['domain']}

Compare it against these existing decisions:
{similar_text}

Return ONLY valid JSON list, no extra text:
[
  {{
    "conflicting_decision": "...",
    "conflict_type": "Direct Contradiction / Dependency / Redundancy / None",
    "explanation": "...",
    "severity": "High / Medium / Low"
  }}
]
Only include entries where conflict_type is NOT None."""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        },
        timeout=60
    )

    resp_json = response.json()
    if "choices" not in resp_json:
        print("❌ API ERROR:", resp_json)
        return []

    raw = resp_json["choices"][0]["message"]["content"]
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    try:
        return json.loads(raw)
    except:
        return []


# ── TEST ──────────────────────────────────
test_decision = {
    "decision": "Increase external hiring by 30% in Q3 2025",
    "rationale": "Need more engineers for product expansion",
    "domain": "HR",
    "owner": "CHRO",
    "deadline": "July 15th",
    "keywords": ["hiring", "expansion", "Q3 2025"],
    "assumptions": ""
}

conflicts = detect_contradictions(test_decision)
print("\n⚠️ Conflicts Detected:")
for c in conflicts:
    print(f"\n🔴 [{c['severity']}] {c['conflict_type']}")
    print(f"   Against: {c['conflicting_decision']}")
    print(f"   Reason : {c['explanation']}")

In [ ]:
# ============================================
# MODULE 4: Decision Debt Scorer
# ============================================

def calculate_debt_score(decision_obj: dict, conflicts: list) -> dict:

    # Rule-based scoring
    score = 0
    reasons = []

    # Factor 1: Conflict severity
    for c in conflicts:
        if c.get("severity") == "High":
            score += 40
            reasons.append("High severity contradiction exists")
        elif c.get("severity") == "Medium":
            score += 20
            reasons.append("Medium severity conflict exists")
        elif c.get("severity") == "Low":
            score += 10
            reasons.append("Low severity dependency exists")

    # Factor 2: Missing owner
    if not decision_obj.get("owner") or decision_obj["owner"] == "N/A":
        score += 15
        reasons.append("No clear owner assigned")

    # Factor 3: Missing deadline
    if not decision_obj.get("deadline") or decision_obj["deadline"] == "N/A":
        score += 15
        reasons.append("No deadline defined")

    # Factor 4: No assumptions documented
    if not decision_obj.get("assumptions") or decision_obj["assumptions"] == "":
        score += 10
        reasons.append("No assumptions documented")

    # Factor 5: Cross-domain risk
    conflict_domains = [c.get("conflict_type") for c in conflicts]
    if "Direct Contradiction" in conflict_domains:
        score += 20
        reasons.append("Direct contradiction increases execution risk")

    score = min(score, 100)

    # Risk label
    if score >= 70:
        label = "🔴 Critical"
    elif score >= 40:
        label = "🟡 Moderate"
    else:
        label = "🟢 Low"

    return {
        "debt_score": score,
        "risk_label": label,
        "reasons": reasons
    }


# ── TEST ──────────────────────────────────
debt = calculate_debt_score(test_decision, conflicts)
print(f"\n📊 Decision Debt Score: {debt['debt_score']}/100")
print(f"🏷️  Risk Level: {debt['risk_label']}")
print(f"\n📋 Reasons:")
for r in debt["reasons"]:
    print(f"   • {r}")

In [ ]:
# ============================================
# MODULE 5: Decision Graph Builder
# ============================================

!pip install pyvis -q

from pyvis.network import Network
import json

def build_decision_graph(stored_ids: list, new_decision_obj: dict, conflicts: list, debt: dict) -> str:
    net = Network(height="580px", width="100%", bgcolor="#0a0a14", font_color="white", directed=True)
    net.barnes_hut()

    domain_colors = {
        "HR": "#4e9af1", "Finance": "#f1c40f",
        "Tech": "#2ecc71", "Operations": "#e67e22",
        "Strategy": "#9b59b6", "Other": "#95a5a6"
    }

    all_decisions = collection.get()
    ids = all_decisions["ids"]
    metas = all_decisions["metadatas"]

    for did, meta in zip(ids, metas):
        color = domain_colors.get(meta.get("domain", "Other"), "#95a5a6")
        net.add_node(did,
            label=f"{meta['decision'][:35]}...",
            color=color, size=25,
            title=f"<b>{meta['decision']}</b><br>Owner: {meta.get('owner','N/A')}<br>Deadline: {meta.get('deadline','N/A')}<br>Domain: {meta.get('domain','N/A')}")

    new_id = "NEW"
    score = debt["debt_score"]
    new_color = "#ff4757" if score >= 70 else "#ffa502" if score >= 40 else "#2ed573"
    net.add_node(new_id,
        label=f"🆕 {new_decision_obj['decision'][:35]}",
        color=new_color, size=40,
        title=f"<b>NEW DECISION</b><br>Debt Score: {score}/100<br>Risk: {debt['risk_label']}")

    conflict_decisions = [c.get("conflicting_decision", "") for c in conflicts]
    for did, meta in zip(ids, metas):
        for cd in conflict_decisions:
            if cd.lower() in meta.get("decision", "").lower() or cd == did:
                net.add_edge(new_id, did, color="#ff4757", width=3, title="⚠️ Contradiction")
        if meta.get("domain") == new_decision_obj.get("domain"):
            net.add_edge(new_id, did, color="#4e9af1", width=1, title="Same Domain")

    # Force inline rendering — no file needed
    net.save_graph("/content/decision_graph.html")
    html = open("/content/decision_graph.html").read()

    # Inject sandbox escape
    html = html.replace(
        "<head>",
        "<head><base target='_self'>"
    ).replace(
        "<body>",
        "<body style='background:#0a0a14; margin:0; padding:0;'>"
    )
    return html

# ── TEST ──────────────────────────────────
graph = build_decision_graph(stored_ids, test_decision, conflicts, debt)

In [ ]:
# ============================================
# MODULE 6: COMPLETE UPGRADED UI (FIXED)
# ============================================

!pip install gradio plotly -q

import gradio as gr
import plotly.graph_objects as go
import json
import math

# ── REPLACE build_decision_graph with Plotly version ──
def build_decision_graph(stored_ids, new_decision_obj, conflicts, debt):
    all_decisions = collection.get()
    ids = all_decisions["ids"]
    metas = all_decisions["metadatas"]

    domain_colors = {
        "HR": "#4e9af1", "Finance": "#f1c40f",
        "Tech": "#2ecc71", "Operations": "#e67e22",
        "Strategy": "#9b59b6", "Other": "#95a5a6"
    }

    # Layout nodes in a circle
    n = len(ids) + 1
    positions = {}
    for i, did in enumerate(ids):
        angle = 2 * math.pi * i / max(n - 1, 1)
        positions[did] = (math.cos(angle) * 2, math.sin(angle) * 2)
    positions["NEW"] = (0, 0)

    conflict_decisions = [c.get("conflicting_decision", "") for c in conflicts]

    edge_x, edge_y, edge_colors = [], [], []
    for did, meta in zip(ids, metas):
        x0, y0 = positions["NEW"]
        x1, y1 = positions[did]
        is_conflict = any(cd.lower() in meta.get("decision", "").lower() for cd in conflict_decisions)
        color = "#ff4757" if is_conflict else "#4e9af1"
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        edge_colors.append(color)

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=2, color='#4e9af1'),
        hoverinfo='none',
        showlegend=False
    )

    # Conflict edges in red
    red_x, red_y = [], []
    for did, meta in zip(ids, metas):
        is_conflict = any(cd.lower() in meta.get("decision", "").lower() for cd in conflict_decisions)
        if is_conflict:
            x0, y0 = positions["NEW"]
            x1, y1 = positions[did]
            red_x += [x0, x1, None]
            red_y += [y0, y1, None]

    conflict_trace = go.Scatter(
        x=red_x, y=red_y,
        mode='lines',
        line=dict(width=3, color='#ff4757'),
        hoverinfo='none',
        showlegend=False
    )

    # Existing nodes
    node_x = [positions[d][0] for d in ids]
    node_y = [positions[d][1] for d in ids]
    node_colors = [domain_colors.get(m.get("domain", "Other"), "#95a5a6") for m in metas]
    node_text = [f"{m['decision'][:40]}...<br>Owner: {m.get('owner','N/A')}<br>Domain: {m.get('domain','N/A')}" for m in metas]
    node_labels = [m['decision'][:20] + "..." for m in metas]

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        marker=dict(size=28, color=node_colors,
                    line=dict(width=2, color='rgba(255,255,255,0.2)')),
        text=node_labels,
        textposition="bottom center",
        textfont=dict(color='#c9d1d9', size=10),
        hovertext=node_text,
        hoverinfo='text',
        showlegend=False
    )

    # New decision node
    score = debt["debt_score"]
    new_color = "#ff4757" if score >= 70 else "#ffa502" if score >= 40 else "#2ed573"
    new_trace = go.Scatter(
        x=[0], y=[0],
        mode='markers+text',
        marker=dict(size=45, color=new_color,
                    line=dict(width=3, color='white'),
                    symbol='star'),
        text=[f"🆕 NEW<br>{score}/100"],
        textposition="top center",
        textfont=dict(color='white', size=11),
        hovertext=[f"NEW: {new_decision_obj['decision']}<br>Debt: {score}/100<br>Risk: {debt['risk_label']}"],
        hoverinfo='text',
        showlegend=False
    )

    fig = go.Figure(
        data=[edge_trace, conflict_trace, node_trace, new_trace],
        layout=go.Layout(
            paper_bgcolor='#0a0a14',
            plot_bgcolor='#0a0a14',
            margin=dict(l=20, r=20, t=20, b=20),
            height=500,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            font=dict(color='#c9d1d9'),
            annotations=[
                dict(x=0.01, y=0.99, xref='paper', yref='paper',
                     text="🔴 Conflict  🔵 Dependency  ⭐ New Decision",
                     showarrow=False, font=dict(size=11, color='#6b7280'),
                     align='left')
            ]
        )
    )
    return fig


CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300;400;600;700;800&family=JetBrains+Mono:wght@400;600&display=swap');

* { margin: 0; padding: 0; box-sizing: border-box; }

body, .gradio-container {
    background: #06060f !important;
    font-family: 'Space Grotesk', sans-serif !important;
    color: #e0e0ff !important;
}

.gradio-container { max-width: 1400px !important; margin: 0 auto !important; }

/* Animated grid */
body::before {
    content: '';
    position: fixed; top: 0; left: 0;
    width: 100%; height: 100%;
    background-image:
        linear-gradient(rgba(102,126,234,0.04) 1px, transparent 1px),
        linear-gradient(90deg, rgba(102,126,234,0.04) 1px, transparent 1px);
    background-size: 60px 60px;
    animation: gridScroll 25s linear infinite;
    pointer-events: none; z-index: 0;
}
@keyframes gridScroll {
    from { background-position: 0 0; }
    to { background-position: 0 60px; }
}

/* Glow orbs */
body::after {
    content: '';
    position: fixed; top: 0; left: 0;
    width: 100%; height: 100%;
    background:
        radial-gradient(ellipse 600px 400px at 15% 25%, rgba(102,126,234,0.12) 0%, transparent 70%),
        radial-gradient(ellipse 400px 600px at 85% 75%, rgba(118,75,162,0.1) 0%, transparent 70%);
    pointer-events: none; z-index: 0;
    animation: orbPulse 8s ease-in-out infinite alternate;
}
@keyframes orbPulse {
    from { opacity: 0.6; }
    to { opacity: 1; }
}

/* Header */
.app-header {
    text-align: center;
    padding: 52px 20px 28px;
    position: relative; z-index: 1;
}
.app-badge {
    display: inline-flex; align-items: center; gap: 6px;
    background: rgba(102,126,234,0.12);
    border: 1px solid rgba(102,126,234,0.3);
    color: #a78bfa; padding: 5px 18px;
    border-radius: 100px; font-size: 0.72em;
    font-weight: 600; letter-spacing: 3px;
    text-transform: uppercase; margin-bottom: 20px;
    animation: badgeGlow 3s ease-in-out infinite;
}
@keyframes badgeGlow {
    0%,100% { box-shadow: 0 0 0 0 rgba(102,126,234,0); }
    50% { box-shadow: 0 0 25px 2px rgba(102,126,234,0.25); }
}
.app-title {
    font-size: 3.6em; font-weight: 800;
    background: linear-gradient(135deg, #818cf8 0%, #c084fc 40%, #34d399 100%);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    background-clip: text; letter-spacing: -2px; line-height: 1;
    animation: titleShimmer 4s ease-in-out infinite alternate;
}
@keyframes titleShimmer {
    from { filter: brightness(1) saturate(1); }
    to { filter: brightness(1.15) saturate(1.2); }
}
.app-subtitle {
    color: #4b5563; font-size: 1em;
    margin-top: 10px; font-weight: 300; letter-spacing: 0.3px;
}

/* Stats */
.stats-row {
    display: flex; justify-content: center;
    gap: 0; margin: 0 24px 28px;
    background: rgba(255,255,255,0.02);
    border: 1px solid rgba(102,126,234,0.12);
    border-radius: 16px; overflow: hidden;
    position: relative; z-index: 1;
}
.stat-cell {
    flex: 1; padding: 18px 24px; text-align: center;
    border-right: 1px solid rgba(102,126,234,0.1);
    transition: background 0.3s;
}
.stat-cell:last-child { border-right: none; }
.stat-cell:hover { background: rgba(102,126,234,0.05); }
.stat-num {
    font-size: 1.6em; font-weight: 800;
    background: linear-gradient(135deg, #818cf8, #34d399);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    font-family: 'JetBrains Mono', monospace;
}
.stat-lbl { font-size: 0.68em; color: #374151; text-transform: uppercase; letter-spacing: 1.5px; margin-top: 2px; }

/* Panels */
.block { position: relative; z-index: 1; }
.panel-wrap {
    background: rgba(255,255,255,0.018) !important;
    border: 1px solid rgba(102,126,234,0.12) !important;
    border-radius: 20px !important;
    backdrop-filter: blur(12px) !important;
    transition: border-color 0.4s, box-shadow 0.4s !important;
}
.panel-wrap:hover {
    border-color: rgba(102,126,234,0.28) !important;
    box-shadow: 0 0 40px rgba(102,126,234,0.06) !important;
}

/* Textarea */
textarea {
    background: rgba(255,255,255,0.025) !important;
    border: 1px solid rgba(102,126,234,0.18) !important;
    border-radius: 14px !important;
    color: #dde2ff !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-size: 0.94em !important;
    line-height: 1.6 !important;
    transition: border-color 0.3s, box-shadow 0.3s !important;
    resize: vertical !important;
}
textarea:focus {
    border-color: rgba(129,140,248,0.5) !important;
    box-shadow: 0 0 0 3px rgba(129,140,248,0.08), 0 0 20px rgba(129,140,248,0.12) !important;
    outline: none !important;
}

/* Button */
button.primary {
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    border: none !important; border-radius: 14px !important;
    font-weight: 700 !important; font-size: 0.95em !important;
    letter-spacing: 0.5px !important; padding: 14px !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 4px 20px rgba(99,102,241,0.25) !important;
}
button.primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 35px rgba(99,102,241,0.4) !important;
    filter: brightness(1.1) !important;
}
button.primary:active { transform: translateY(0) !important; }

/* Labels */
label span { color: #4b5563 !important; font-size: 0.72em !important; font-weight: 600 !important; text-transform: uppercase !important; letter-spacing: 1.8px !important; }

/* Markdown */
.prose { color: #c9d1d9 !important; }
.prose h2 { color: #818cf8 !important; font-size: 0.82em !important; text-transform: uppercase !important; letter-spacing: 1.5px !important; border-bottom: 1px solid rgba(102,126,234,0.15) !important; padding-bottom: 10px !important; margin-bottom: 14px !important; font-weight: 700 !important; }
.prose h1 { font-size: 2.4em !important; font-weight: 800 !important; font-family: 'JetBrains Mono', monospace !important; }
.prose table { width: 100% !important; border-collapse: collapse !important; font-size: 0.88em !important; }
.prose td, .prose th { padding: 9px 14px !important; border: 1px solid rgba(102,126,234,0.1) !important; }
.prose th { background: rgba(99,102,241,0.08) !important; color: #818cf8 !important; font-size: 0.75em !important; text-transform: uppercase !important; letter-spacing: 1px !important; }
.prose code { background: rgba(99,102,241,0.12) !important; color: #34d399 !important; padding: 2px 8px !important; border-radius: 6px !important; font-family: 'JetBrains Mono', monospace !important; }

/* Loading pulse on outputs */
.generating > * { animation: contentPulse 1.5s ease-in-out infinite !important; }
@keyframes contentPulse {
    0%,100% { opacity: 1; } 50% { opacity: 0.5; }
}

/* Scrollbar */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: rgba(99,102,241,0.25); border-radius: 10px; }
::-webkit-scrollbar-thumb:hover { background: rgba(99,102,241,0.45); }

/* Examples */
.gr-samples-table { background: rgba(255,255,255,0.01) !important; border-radius: 12px !important; border: 1px solid rgba(102,126,234,0.08) !important; }

/* Section label */
.section-label {
    color: #374151; font-size: 0.7em; font-weight: 700;
    text-transform: uppercase; letter-spacing: 2px;
    padding: 0 4px 10px; border-bottom: 1px solid rgba(102,126,234,0.1);
    margin-bottom: 12px;
}
"""

JS = """
function() {
    const style = document.createElement('style');
    style.textContent = `
        @keyframes fadeUp {
            from { opacity:0; transform:translateY(16px); }
            to   { opacity:1; transform:translateY(0); }
        }
        .prose { animation: fadeUp 0.4s ease forwards; }
    `;
    document.head.appendChild(style);
}
"""

def process_decision(text: str):
    if not text.strip():
        return "❌ Please enter a decision.", "", "", None

    decision_obj = extract_decision(text)
    did = store_decision(decision_obj)
    conflicts = detect_contradictions(decision_obj)
    debt = calculate_debt_score(decision_obj, conflicts)
    fig = build_decision_graph(stored_ids, decision_obj, conflicts, debt)
    stored_ids.append(did)

    decision_md = f"""## 📋 Extracted Decision
| Field | Value |
|-------|-------|
| **Decision** | {decision_obj['decision']} |
| **Rationale** | {decision_obj['rationale']} |
| **Owner** | {decision_obj.get('owner','N/A')} |
| **Deadline** | {decision_obj.get('deadline','N/A')} |
| **Domain** | {decision_obj.get('domain','N/A')} |
| **Keywords** | {', '.join(decision_obj.get('keywords', [])) if isinstance(decision_obj.get('keywords'), list) else decision_obj.get('keywords','N/A')} |
"""

    conflict_md = "## ⚠️ Conflict Analysis\n"
    if conflicts:
        for c in conflicts:
            icon = "🔴" if c['severity']=="High" else "🟡" if c['severity']=="Medium" else "🟢"
            conflict_md += f"\n{icon} **[{c['severity']}] {c['conflict_type']}**\n- **Against:** {c['conflicting_decision']}\n- **Reason:** {c['explanation']}\n\n---\n"
    else:
        conflict_md += "\n✅ **No conflicts detected.** This decision is clean."

    score = debt['debt_score']
    filled = round(score / 10)
    bar = "█" * filled + "░" * (10 - filled)
    debt_md = f"""## 📊 Decision Debt Score

# {score}/100 — {debt['risk_label']}

`{bar}` {score}%

**Contributing Factors:**
""" + "\n".join(f"- {r}" for r in debt["reasons"])

    return decision_md, conflict_md, debt_md, fig


# ── UI ────────────────────────────────────
with gr.Blocks(css=CSS, js=JS, title="⚡ Decision Debt Tracker") as demo:

    gr.HTML("""
    <div class='app-header'>
        <div class='app-badge'>⚡ Corporate Intelligence Platform</div>
        <div class='app-title'>Decision Debt Tracker</div>
        <div class='app-subtitle'>Detect conflicts · Score risk · Visualize organizational decision impact</div>
    </div>
    <div class='stats-row'>
        <div class='stat-cell'><div class='stat-num'>LLM</div><div class='stat-lbl'>AI Engine</div></div>
        <div class='stat-cell'><div class='stat-num'>Real-Time</div><div class='stat-lbl'>Conflict Detection</div></div>
        <div class='stat-cell'><div class='stat-num'>Graph</div><div class='stat-lbl'>Decision Network</div></div>
        <div class='stat-cell'><div class='stat-num'>100pt</div><div class='stat-lbl'>Debt Scoring</div></div>
    </div>
    """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, elem_classes="panel-wrap"):
            inp = gr.Textbox(
                label="📝 Decision Input",
                placeholder="Enter a corporate decision or paste a meeting transcript...\n\ne.g. The board decided to freeze all external hiring for Q3 2025 due to budget overruns. CFO owns this. Deadline: July 1st.",
                lines=9
            )
            btn = gr.Button("⚡ Analyze Decision", variant="primary", size="lg")
            gr.Examples(
                examples=[
                    ["The board decided to expand into Southeast Asia by Q3 2025, hiring 200 new staff. CEO owns this."],
                    ["Finance has frozen all travel budgets company-wide effective immediately due to Q2 losses. CFO owns this."],
                    ["Engineering will migrate to fully remote infrastructure, eliminating all on-premise servers by Dec 2025."],
                    ["HR approved a 30% salary hike across all departments to retain talent, approved by CHRO, effective Q3."]
                ],
                inputs=inp,
                label="💡 Quick Examples"
            )

        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column(elem_classes="panel-wrap"):
                    out_decision = gr.Markdown(elem_classes="prose")
                with gr.Column(elem_classes="panel-wrap"):
                    out_conflict = gr.Markdown(elem_classes="prose")
            with gr.Column(elem_classes="panel-wrap"):
                out_debt = gr.Markdown(elem_classes="prose")

    with gr.Column(elem_classes="panel-wrap"):
        gr.HTML("<div class='section-label'>🕸️ Decision Network Graph</div>")
        out_graph = gr.Plot(label="")

    btn.click(
        fn=process_decision,
        inputs=inp,
        outputs=[out_decision, out_conflict, out_debt, out_graph]
    )

demo.launch(share=True)

In [ ]:
app_code = '''
import os
import requests
import json
import math
import uuid
import gradio as gr
import plotly.graph_objects as go
import chromadb
from sentence_transformers import SentenceTransformer

# ── CONFIG ────────────────────────────────
GROQ_API_KEY = os.environ.get("GROQ_NEW_API_KEY", "").strip()
MODEL = "llama-3.3-70b-versatile"

# ── MODULE 2: INIT ────────────────────────
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("decisions")
stored_ids = []

# ── MODULE 1: EXTRACTOR ───────────────────
def extract_decision(text: str) -> dict:
    prompt = f"""You are a corporate decision intelligence engine.
Extract structured decision data from the following text.
Return ONLY valid JSON, no extra text:
{{
  "decision": "core decision made",
  "rationale": "why this decision was made",
  "assumptions": ["assumption1", "assumption2"],
  "owner": "person/team responsible",
  "deadline": "deadline if mentioned, else null",
  "domain": "HR/Finance/Tech/Operations/Strategy/Other",
  "keywords": ["kw1", "kw2", "kw3"]
}}
Text: {text}"""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
        json={"model": MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0.2},
        timeout=60
    )
    resp_json = response.json()
    if "choices" not in resp_json:
        raise ValueError(f"API Error: {resp_json}")
    raw = resp_json["choices"][0]["message"]["content"]
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

# ── MODULE 2: STORE ───────────────────────
def flatten_metadata(obj: dict) -> dict:
    return {k: (", ".join(v) if isinstance(v, list) else (v or "N/A")) for k, v in obj.items()}

def store_decision(decision_obj: dict) -> str:
    decision_id = str(uuid.uuid4())
    text_to_embed = f"{decision_obj[\'decision\']}. {decision_obj[\'rationale\']}. {\' \'.join(decision_obj[\'keywords\']) if isinstance(decision_obj[\'keywords\'], list) else decision_obj[\'keywords\']}"
    embedding = embedder.encode(text_to_embed).tolist()
    collection.add(ids=[decision_id], embeddings=[embedding], documents=[text_to_embed], metadatas=[flatten_metadata(decision_obj)])
    return decision_id

def retrieve_similar(query: str, top_k: int = 5) -> list:
    if collection.count() == 0:
        return []
    top_k = min(top_k, collection.count())
    embedding = embedder.encode(query).tolist()
    results = collection.query(query_embeddings=[embedding], n_results=top_k)
    return results["metadatas"][0]

# ── MODULE 3: CONTRADICTIONS ──────────────
def detect_contradictions(new_decision_obj: dict) -> list:
    similar = retrieve_similar(f"{new_decision_obj[\'decision\']}. {new_decision_obj[\'rationale\']}", top_k=5)
    if not similar:
        return []
    similar_text = "\\n".join([
        f"- [{i+1}] Decision: {d[\'decision\']} | Rationale: {d[\'rationale\']} | Domain: {d[\'domain\']} | Owner: {d[\'owner\']}"
        for i, d in enumerate(similar)
    ])
    prompt = f"""You are a corporate decision auditor.
A new decision has been made:
Decision: {new_decision_obj[\'decision\']}
Rationale: {new_decision_obj[\'rationale\']}
Domain: {new_decision_obj[\'domain\']}

Compare it against these existing decisions:
{similar_text}

Return ONLY valid JSON list, no extra text:
[{{"conflicting_decision": "...","conflict_type": "Direct Contradiction / Dependency / Redundancy / None","explanation": "...","severity": "High / Medium / Low"}}]
Only include entries where conflict_type is NOT None."""

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
        json={"model": MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0.2},
        timeout=60
    )
    resp_json = response.json()
    if "choices" not in resp_json:
        return []
    raw = resp_json["choices"][0]["message"]["content"]
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except:
        return []

# ── MODULE 4: DEBT SCORER ─────────────────
def calculate_debt_score(decision_obj: dict, conflicts: list) -> dict:
    score = 0
    reasons = []
    for c in conflicts:
        if c.get("severity") == "High": score += 40; reasons.append("High severity contradiction exists")
        elif c.get("severity") == "Medium": score += 20; reasons.append("Medium severity conflict exists")
        elif c.get("severity") == "Low": score += 10; reasons.append("Low severity dependency exists")
    if not decision_obj.get("owner") or decision_obj["owner"] == "N/A": score += 15; reasons.append("No clear owner assigned")
    if not decision_obj.get("deadline") or decision_obj["deadline"] == "N/A": score += 15; reasons.append("No deadline defined")
    if not decision_obj.get("assumptions") or decision_obj["assumptions"] == "": score += 10; reasons.append("No assumptions documented")
    if any(c.get("conflict_type") == "Direct Contradiction" for c in conflicts): score += 20; reasons.append("Direct contradiction increases execution risk")
    score = min(score, 100)
    label = "🔴 Critical" if score >= 70 else "🟡 Moderate" if score >= 40 else "🟢 Low"
    return {"debt_score": score, "risk_label": label, "reasons": reasons}

# ── MODULE 5: GRAPH ───────────────────────
def build_decision_graph(stored_ids, new_decision_obj, conflicts, debt):
    all_decisions = collection.get()
    ids = all_decisions["ids"]
    metas = all_decisions["metadatas"]
    domain_colors = {"HR": "#4e9af1","Finance": "#f1c40f","Tech": "#2ecc71","Operations": "#e67e22","Strategy": "#9b59b6","Other": "#95a5a6"}
    n = len(ids) + 1
    positions = {}
    for i, did in enumerate(ids):
        angle = 2 * math.pi * i / max(n - 1, 1)
        positions[did] = (math.cos(angle) * 2, math.sin(angle) * 2)
    positions["NEW"] = (0, 0)
    conflict_decisions = [c.get("conflicting_decision", "") for c in conflicts]
    edge_x, edge_y, red_x, red_y = [], [], [], []
    for did, meta in zip(ids, metas):
        x0, y0 = positions["NEW"]; x1, y1 = positions[did]
        is_conflict = any(cd.lower() in meta.get("decision", "").lower() for cd in conflict_decisions)
        if is_conflict: red_x += [x0,x1,None]; red_y += [y0,y1,None]
        else: edge_x += [x0,x1,None]; edge_y += [y0,y1,None]
    score = debt["debt_score"]
    new_color = "#ff4757" if score >= 70 else "#ffa502" if score >= 40 else "#2ed573"
    fig = go.Figure(
        data=[
            go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1.5, color="#4e9af1"), hoverinfo="none", showlegend=False),
            go.Scatter(x=red_x, y=red_y, mode="lines", line=dict(width=3, color="#ff4757"), hoverinfo="none", showlegend=False),
            go.Scatter(
                x=[positions[d][0] for d in ids], y=[positions[d][1] for d in ids],
                mode="markers+text",
                marker=dict(size=28, color=[domain_colors.get(m.get("domain","Other"),"#95a5a6") for m in metas], line=dict(width=2, color="rgba(255,255,255,0.2)")),
                text=[m["decision"][:20]+"..." for m in metas],
                textposition="bottom center", textfont=dict(color="#c9d1d9", size=10),
                hovertext=[f"{m[\'decision\']}<br>Owner: {m.get(\'owner\',\'N/A\')}<br>Domain: {m.get(\'domain\',\'N/A\')}" for m in metas],
                hoverinfo="text", showlegend=False
            ),
            go.Scatter(
                x=[0], y=[0], mode="markers+text",
                marker=dict(size=45, color=new_color, line=dict(width=3, color="white"), symbol="star"),
                text=[f"🆕 NEW\\n{score}/100"], textposition="top center",
                textfont=dict(color="white", size=11),
                hovertext=[f"NEW: {new_decision_obj[\'decision\']}<br>Debt: {score}/100<br>Risk: {debt[\'risk_label\']}"],
                hoverinfo="text", showlegend=False
            )
        ],
        layout=go.Layout(
            paper_bgcolor="#0a0a14", plot_bgcolor="#0a0a14",
            margin=dict(l=20,r=20,t=20,b=20), height=500,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            font=dict(color="#c9d1d9"),
            annotations=[dict(x=0.01,y=0.99,xref="paper",yref="paper",
                text="🔴 Conflict  🔵 Dependency  ⭐ New Decision",
                showarrow=False, font=dict(size=11, color="#6b7280"))]
        )
    )
    return fig

# ── MODULE 6: PROCESS ─────────────────────
def process_decision(text: str):
    if not text.strip():
        return "❌ Please enter a decision.", "", "", None
    decision_obj = extract_decision(text)
    did = store_decision(decision_obj)
    conflicts = detect_contradictions(decision_obj)
    debt = calculate_debt_score(decision_obj, conflicts)
    fig = build_decision_graph(stored_ids, decision_obj, conflicts, debt)
    stored_ids.append(did)
    decision_md = f"""## 📋 Extracted Decision
| Field | Value |
|-------|-------|
| **Decision** | {decision_obj[\'decision\']} |
| **Rationale** | {decision_obj[\'rationale\']} |
| **Owner** | {decision_obj.get(\'owner\',\'N/A\')} |
| **Deadline** | {decision_obj.get(\'deadline\',\'N/A\')} |
| **Domain** | {decision_obj.get(\'domain\',\'N/A\')} |
| **Keywords** | {\', \'.join(decision_obj.get(\'keywords\', [])) if isinstance(decision_obj.get(\'keywords\'), list) else decision_obj.get(\'keywords\',\'N/A\')} |
"""
    conflict_md = "## ⚠️ Conflict Analysis\\n"
    if conflicts:
        for c in conflicts:
            icon = "🔴" if c[\'severity\']=="High" else "🟡" if c[\'severity\']=="Medium" else "🟢"
            conflict_md += f"\\n{icon} **[{c[\'severity\']}] {c[\'conflict_type\']}**\\n- **Against:** {c[\'conflicting_decision\']}\\n- **Reason:** {c[\'explanation\']}\\n\\n---\\n"
    else:
        conflict_md += "\\n✅ **No conflicts detected.** This decision is clean."
    score = debt[\'debt_score\']
    bar = "█" * round(score/10) + "░" * (10 - round(score/10))
    debt_md = f"""## 📊 Decision Debt Score\\n\\n# {score}/100 — {debt[\'risk_label\']}\\n\\n`{bar}` {score}%\\n\\n**Contributing Factors:**\\n""" + "\\n".join(f"- {r}" for r in debt["reasons"])
    return decision_md, conflict_md, debt_md, fig

# ── UI ────────────────────────────────────
CSS = """
@import url(\'https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300;400;600;700;800&family=JetBrains+Mono:wght@400;600&display=swap\');
* { margin:0; padding:0; box-sizing:border-box; }
body, .gradio-container { background:#06060f !important; font-family:\'Space Grotesk\',sans-serif !important; color:#e0e0ff !important; }
.gradio-container { max-width:1400px !important; margin:0 auto !important; }
body::before { content:\'\'; position:fixed; top:0; left:0; width:100%; height:100%;
    background-image: linear-gradient(rgba(102,126,234,0.04) 1px,transparent 1px), linear-gradient(90deg,rgba(102,126,234,0.04) 1px,transparent 1px);
    background-size:60px 60px; animation:gridScroll 25s linear infinite; pointer-events:none; z-index:0; }
@keyframes gridScroll { from{background-position:0 0} to{background-position:0 60px} }
body::after { content:\'\'; position:fixed; top:0; left:0; width:100%; height:100%;
    background: radial-gradient(ellipse 600px 400px at 15% 25%,rgba(102,126,234,0.12) 0%,transparent 70%),
                radial-gradient(ellipse 400px 600px at 85% 75%,rgba(118,75,162,0.1) 0%,transparent 70%);
    pointer-events:none; z-index:0; animation:orbPulse 8s ease-in-out infinite alternate; }
@keyframes orbPulse { from{opacity:0.6} to{opacity:1} }
.app-header { text-align:center; padding:52px 20px 28px; position:relative; z-index:1; }
.app-badge { display:inline-flex; align-items:center; gap:6px; background:rgba(102,126,234,0.12); border:1px solid rgba(102,126,234,0.3); color:#a78bfa; padding:5px 18px; border-radius:100px; font-size:0.72em; font-weight:600; letter-spacing:3px; text-transform:uppercase; margin-bottom:20px; animation:badgeGlow 3s ease-in-out infinite; }
@keyframes badgeGlow { 0%,100%{box-shadow:0 0 0 0 rgba(102,126,234,0)} 50%{box-shadow:0 0 25px 2px rgba(102,126,234,0.25)} }
.app-title { font-size:3.6em; font-weight:800; background:linear-gradient(135deg,#818cf8 0%,#c084fc 40%,#34d399 100%); -webkit-background-clip:text; -webkit-text-fill-color:transparent; background-clip:text; letter-spacing:-2px; animation:titleShimmer 4s ease-in-out infinite alternate; }
@keyframes titleShimmer { from{filter:brightness(1)} to{filter:brightness(1.15)} }
.app-subtitle { color:#4b5563; font-size:1em; margin-top:10px; font-weight:300; }
.stats-row { display:flex; justify-content:center; margin:0 24px 28px; background:rgba(255,255,255,0.02); border:1px solid rgba(102,126,234,0.12); border-radius:16px; overflow:hidden; position:relative; z-index:1; }
.stat-cell { flex:1; padding:18px 24px; text-align:center; border-right:1px solid rgba(102,126,234,0.1); transition:background 0.3s; }
.stat-cell:last-child { border-right:none; }
.stat-cell:hover { background:rgba(102,126,234,0.05); }
.stat-num { font-size:1.6em; font-weight:800; background:linear-gradient(135deg,#818cf8,#34d399); -webkit-background-clip:text; -webkit-text-fill-color:transparent; font-family:\'JetBrains Mono\',monospace; }
.stat-lbl { font-size:0.68em; color:#374151; text-transform:uppercase; letter-spacing:1.5px; margin-top:2px; }
.panel-wrap { background:rgba(255,255,255,0.018) !important; border:1px solid rgba(102,126,234,0.12) !important; border-radius:20px !important; backdrop-filter:blur(12px) !important; transition:border-color 0.4s,box-shadow 0.4s !important; }
.panel-wrap:hover { border-color:rgba(102,126,234,0.28) !important; box-shadow:0 0 40px rgba(102,126,234,0.06) !important; }
textarea { background:rgba(255,255,255,0.025) !important; border:1px solid rgba(102,126,234,0.18) !important; border-radius:14px !important; color:#dde2ff !important; font-family:\'Space Grotesk\',sans-serif !important; font-size:0.94em !important; }
textarea:focus { border-color:rgba(129,140,248,0.5) !important; box-shadow:0 0 0 3px rgba(129,140,248,0.08) !important; outline:none !important; }
button.primary { background:linear-gradient(135deg,#6366f1,#8b5cf6) !important; border:none !important; border-radius:14px !important; font-weight:700 !important; padding:14px !important; transition:all 0.3s !important; box-shadow:0 4px 20px rgba(99,102,241,0.25) !important; }
button.primary:hover { transform:translateY(-2px) !important; box-shadow:0 8px 35px rgba(99,102,241,0.4) !important; }
label span { color:#4b5563 !important; font-size:0.72em !important; font-weight:600 !important; text-transform:uppercase !important; letter-spacing:1.8px !important; }
.prose h2 { color:#818cf8 !important; font-size:0.82em !important; text-transform:uppercase !important; letter-spacing:1.5px !important; border-bottom:1px solid rgba(102,126,234,0.15) !important; padding-bottom:10px !important; margin-bottom:14px !important; }
.prose h1 { font-size:2.4em !important; font-weight:800 !important; font-family:\'JetBrains Mono\',monospace !important; }
.prose table { width:100% !important; border-collapse:collapse !important; font-size:0.88em !important; }
.prose td,.prose th { padding:9px 14px !important; border:1px solid rgba(102,126,234,0.1) !important; }
.prose th { background:rgba(99,102,241,0.08) !important; color:#818cf8 !important; font-size:0.75em !important; text-transform:uppercase !important; }
.prose code { background:rgba(99,102,241,0.12) !important; color:#34d399 !important; padding:2px 8px !important; border-radius:6px !important; font-family:\'JetBrains Mono\',monospace !important; }
.section-label { color:#374151; font-size:0.7em; font-weight:700; text-transform:uppercase; letter-spacing:2px; padding:0 4px 10px; border-bottom:1px solid rgba(102,126,234,0.1); margin-bottom:12px; }
::-webkit-scrollbar { width:5px; } ::-webkit-scrollbar-track { background:transparent; } ::-webkit-scrollbar-thumb { background:rgba(99,102,241,0.25); border-radius:10px; }
"""

with gr.Blocks(css=CSS, title="⚡ Decision Debt Tracker") as demo:
    gr.HTML("""
    <div class=\'app-header\'>
        <div class=\'app-badge\'>⚡ Corporate Intelligence Platform</div>
        <div class=\'app-title\'>Decision Debt Tracker</div>
        <div class=\'app-subtitle\'>Detect conflicts · Score risk · Visualize organizational decision impact</div>
    </div>
    <div class=\'stats-row\'>
        <div class=\'stat-cell\'><div class=\'stat-num\'>LLM</div><div class=\'stat-lbl\'>AI Engine</div></div>
        <div class=\'stat-cell\'><div class=\'stat-num\'>Real-Time</div><div class=\'stat-lbl\'>Conflict Detection</div></div>
        <div class=\'stat-cell\'><div class=\'stat-num\'>Graph</div><div class=\'stat-lbl\'>Decision Network</div></div>
        <div class=\'stat-cell\'><div class=\'stat-num\'>100pt</div><div class=\'stat-lbl\'>Debt Scoring</div></div>
    </div>
    """)
    with gr.Row(equal_height=False):
        with gr.Column(scale=1, elem_classes="panel-wrap"):
            inp = gr.Textbox(label="📝 Decision Input", placeholder="Enter a corporate decision or paste a meeting transcript...", lines=9)
            btn = gr.Button("⚡ Analyze Decision", variant="primary", size="lg")
            gr.Examples(
                examples=[
                    ["The board decided to expand into Southeast Asia by Q3 2025, hiring 200 new staff. CEO owns this."],
                    ["Finance has frozen all travel budgets company-wide effective immediately due to Q2 losses. CFO owns this."],
                    ["Engineering will migrate to fully remote infrastructure, eliminating all on-premise servers by Dec 2025."],
                    ["HR approved a 30% salary hike across all departments to retain talent, approved by CHRO, effective Q3."]
                ],
                inputs=inp, label="💡 Quick Examples"
            )
        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column(elem_classes="panel-wrap"):
                    out_decision = gr.Markdown(elem_classes="prose")
                with gr.Column(elem_classes="panel-wrap"):
                    out_conflict = gr.Markdown(elem_classes="prose")
            with gr.Column(elem_classes="panel-wrap"):
                out_debt = gr.Markdown(elem_classes="prose")
    with gr.Column(elem_classes="panel-wrap"):
        gr.HTML("<div class=\'section-label\'>🕸️ Decision Network Graph</div>")
        out_graph = gr.Plot(label="")
    btn.click(fn=process_decision, inputs=inp, outputs=[out_decision, out_conflict, out_debt, out_graph])

demo.launch()
'''

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created successfully")

In [ ]:
!pip install huggingface_hub -q
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()
api = HfApi()

# Upload app.py
api.upload_file(
    path_or_fileobj="/content/app.py",
    path_in_repo="app.py",
    repo_id="swastikraj/decision-debt-tracker",
    repo_type="space",
    token=HF_TOKEN
)

# Upload requirements.txt
reqs = "gradio\nrequests\nchromadb\nsentence-transformers\nplotly\npyvis\n"
with open("/content/requirements.txt", "w") as f:
    f.write(reqs)

api.upload_file(
    path_or_fileobj="/content/requirements.txt",
    path_in_repo="requirements.txt",
    repo_id="swastikraj/decision-debt-tracker",
    repo_type="space",
    token=HF_TOKEN
)

print("✅ Deployed! Visit: https://huggingface.co/spaces/swastikraj/decision-debt-tracker")

In [ ]:
# Clear widget metadata before downloading
import IPython
IPython.display.clear_output()